# 01 — 2D Scan

Define a phase transition model, run the BubbleMaster 2D scan over γ and time,
and cache results for bootstrapping in notebook 02.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ptbuilder as pt

## 1. Define the model

In [ ]:
# Choose potential and create PhysicsModel
lambda_bar = 0.84

potential = pt.Phi4Potential(lambda_bar)
config    = pt.Config()          # uses default data/ directory and bin/ binaries
model     = pt.PhysicsModel(potential, config)

print(f"Model name:   {model.name}")
print(f"Results dir:  {model.results_dir}")
print(f"phi_false:    {potential.phi_false}")
print(f"phi_true:     {potential.phi_true}")
print(f"Delta V:      {potential.delta_V:.6f}")

## 2. Compute the instanton profile (disk-cached)

In [ ]:
profile = model.instanton

print(f"rin_0:   {profile.rin_0:.4f}")
print(f"rmid_0:  {profile.rmid_0:.4f}")
print(f"rout_0:  {profile.rout_0:.4f}")
print(f"wall width: {profile.wall_width:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(profile.R, profile.Phi)
plt.axvline(profile.rin_0,  ls='--', color='C1', label='r_in')
plt.axvline(profile.rmid_0, ls='--', color='C2', label='r_mid')
plt.axvline(profile.rout_0, ls='--', color='C3', label='r_out')
plt.xlabel('r')
plt.ylabel(r'$\phi$')
plt.title('Instanton profile')
plt.legend()
plt.tight_layout()

## 3. Define the scan grid

In [ ]:
n_gamma  = 21
n_t      = 32

gammas   = np.linspace(1., 6., n_gamma)
times    = np.linspace(1., 63., n_t)

scan_cfg = pt.ScanConfig(
    gammas = gammas,
    times  = times,
    bm_params = pt.BubbleMasterParams(
        dz           = 0.005,
        n_w          = 128,
        n_k          = 51,
        how_often_ds = 5,
        baby_steps   = 100,
        cutoff_type  = 0,
    )
)

print(f"Gammas: {gammas[[0,-1]]}  ({n_gamma} points)")
print(f"Times:  {times[[0,-1]]}  ({n_t} points)")

## 4. Run the scan (cache-aware)

Each (γ, model) pair is run once and cached in `data/<model_name>/scan_cache.h5`.
Re-running this cell will only compute missing entries.

In [ ]:
# Make sure the binaries are compiled first (run `make` from repo root)
config.check_binaries()

scan_results = pt.run_scan(model, scan_cfg, config)

print(f"\nScan complete: {len(scan_results)} gamma values")

## 5. Inspect results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

i_gamma_show = 8   # middle gamma
gamma_show   = gammas[i_gamma_show]
res          = scan_results[float(gamma_show)]

# Cumulative spectrum at each time
ax = axes[0]
for i_t in range(n_t):
    ax.plot(res.w, res.spectrum[i_t], alpha=0.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('w')
ax.set_ylabel('spectrum')
ax.set_title(f'Cumulative spectrum  (γ={gamma_show:.1f})')

# dE/dt heatmap
ax = axes[1]
dEdt_show = np.array([scan_results[float(g)].dEdt[i_gamma_show]
                       for g in gammas])
ax.imshow(dEdt_show, aspect='auto', origin='lower', cmap='afmhot')
ax.set_xlabel('k index')
ax.set_ylabel('gamma index')
ax.set_title('dE/dt(γ, k)')

plt.tight_layout()